# Metabolite Availability - CELL MESH Integration Test

这是 cellmesh 包的 metabolite availability 模块集成测试 notebook。

注意：本 notebook 所有生产计算都调用 cellmesh 包内真实方法，不在 notebook 中重新实现算法。

In [62]:
# 1. 导入 cellmesh
import cellmesh
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display

print(f"cellmesh 导入路径: {cellmesh.__file__}")
print(f"cellmesh 版本: {cellmesh.__version__}")
print(f"✓ compute_metabolite_availability 可用: {'compute_metabolite_availability' in dir(cellmesh)}")

cellmesh 导入路径: /home/qsong/.openclaw/workspace/developer/cell_mesh_pkg/cellmesh/__init__.py
cellmesh 版本: 0.4.0
✓ compute_metabolite_availability 可用: True


## 2. 生成 Toy AnnData

按照文档要求，使用指定的基因和细胞类型。

In [63]:
# 2.1 定义基因和细胞类型
genes = [
    "G_prodA1", "G_prodA2", "G_prodA3",
    "G_consA",
    "G_transA1", "G_transA2",
    "G_prodB", "G_prodF",
    "G_noise"
]

cells = ["T1", "T2", "T3", "M1", "M2", "M3", "F1", "F2"]

cell_types = [
    "T_cell", "T_cell", "T_cell",
    "Macrophage", "Macrophage", "Macrophage",
    "Fibroblast", "Fibroblast"
]

# 2.2 构建表达矩阵（已经是 log-normalized）
X = np.array([
    [4.0, 1.0, 0.7, 0.2, 2.5, 1.5, 0.1, 1.2, 0.5],
    [0, 1.2, 0.8, 0.3, 2.7, 1.4, 0.0, 1.1, 0.4],
    [0, 0.8, 0.6, 0.2, 2.4, 1.6, 0.2, 1.3, 0.6],
    [1.0, 2.8, 0.5, 4.0, 0.8, 0.4, 1.0, 0.5, 0.7],
    [1.2, 3.0, 0.5, 4.2, 0.7, 0.3, 0.8, 0.6, 0.8],
    [0.9, 2.6, 0.6, 3.8, 0.9, 0.5, 1.1, 0.4, 0.6],
    [0.5, 0.6, 0.2, 0.8, 0.3, 0.2, 4.0, 2.5, 0.9],
    [0.4, 0.7, 0.3, 0.9, 0.4, 0.2, 4.2, 2.7, 1.0],
], dtype=float)

In [64]:
# 2.3 创建简单的 AnnData 类
class SimpleAnnData:
    def __init__(self, X, var_names, obs):
        self.X = X
        self.layers = {}
        self.var_names = pd.Index(var_names)
        self.obs = pd.DataFrame(obs)

adata = SimpleAnnData(
    X=X,
    var_names=genes,
    obs={"cell_type": cell_types}
)

# 2.4 展示 cell × gene 表达矩阵
print("AnnData 形状:", adata.X.shape)
expr_df = pd.DataFrame(adata.X, index=cells, columns=genes)
expr_df.insert(0, "cell_type", cell_types)
display(expr_df.round(3))

AnnData 形状: (8, 9)


,cell_type,G_prodA1,G_prodA2,G_prodA3,G_consA,G_transA1,G_transA2,G_prodB,G_prodF,G_noise
T1,T_cell,4.0,1.0,0.7,0.2,2.5,1.5,0.1,1.2,0.5
T2,T_cell,0.0,1.2,0.8,0.3,2.7,1.4,0.0,1.1,0.4
T3,T_cell,0.0,0.8,0.6,0.2,2.4,1.6,0.2,1.3,0.6
M1,Macrophage,1.0,2.8,0.5,4.0,0.8,0.4,1.0,0.5,0.7
M2,Macrophage,1.2,3.0,0.5,4.2,0.7,0.3,0.8,0.6,0.8
M3,Macrophage,0.9,2.6,0.6,3.8,0.9,0.5,1.1,0.4,0.6
F1,Fibroblast,0.5,0.6,0.2,0.8,0.3,0.2,4.0,2.5,0.9
F2,Fibroblast,0.4,0.7,0.3,0.9,0.4,0.2,4.2,2.7,1.0


## 3. 调用 cellmesh 构建 Pseudobulk

通过 compute_metabolite_availability 的中间结果来获取 pseudobulk。

In [65]:
# 3.1 先生成一个简单的 reaction table 来调用 API
simple_reaction = pd.DataFrame([
    {"metabolite": "Test", "hmdb_id": "HMDB000", "reaction": "R0", "gene": "G_noise", "direction": "product"}
])

# 3.2 调用 compute_metabolite_availability 获取 pseudobulk
simple_result = cellmesh.compute_metabolite_availability(
    adata,
    simple_reaction,
    celltype_col="cell_type",
    min_cells=1,
    lower=0,
    upper=100,
    return_intermediates=True
)

# 3.3 展示 pseudobulk
print("\nCell type pseudobulk / cell_type × gene:")
pseudobulk = simple_result['pseudobulk']
display(pseudobulk.round(3))


Cell type pseudobulk / cell_type × gene:


,G_prodA1,G_prodA2,G_prodA3,G_consA,G_transA1,G_transA2,G_prodB,G_prodF,G_noise
T_cell,1.333,1.00,0.700,0.233,2.533,1.5,0.100,1.2,0.50
Macrophage,1.033,2.80,0.533,4.000,0.800,0.4,0.967,0.5,0.70
Fibroblast,0.450,0.65,0.250,0.850,0.350,0.2,4.100,2.6,0.95


In [ ]:
simple_result.keys()

In [67]:
simple_result['metadata']

,,has_product,has_substrate,has_exporter,n_product_reactions,n_substrate_reactions,n_exporter_reactions
metabolite,hmdb_id,,,,,,
Test,HMDB000,True,False,False,1,0,0


## 4. 生成 Reaction Table

按照文档要求构建完整的 reaction table。

In [68]:
# 4.1 构建完整的 reaction table
reaction_table = pd.DataFrame([
    {"metabolite": "MetA", "hmdb_id": "HMDB00001", "reaction": "R_A_prod_1", "gene": "G_prodA1", "direction": "product"},
    {"metabolite": "MetA", "hmdb_id": "HMDB00001", "reaction": "R_A_prod_1", "gene": "G_prodA2", "direction": "product"},
    {"metabolite": "MetA", "hmdb_id": "HMDB00001", "reaction": "R_A_prod_2", "gene": "G_prodA3", "direction": "product"},
    {"metabolite": "MetA", "hmdb_id": "HMDB00001", "reaction": "R_A_sub_1", "gene": "G_consA", "direction": "substrate"},
    {"metabolite": "MetA", "hmdb_id": "HMDB00001", "reaction": "R_A_trans_1", "gene": "G_transA1", "direction": "exporter"},
    {"metabolite": "MetA", "hmdb_id": "HMDB00001", "reaction": "R_A_trans_1", "gene": "G_transA2", "direction": "exporter"},
    {"metabolite": "MetB", "hmdb_id": "HMDB00002", "reaction": "R_B_prod_1", "gene": "G_prodB, G_prodA1", "direction": "product"},
    {"metabolite": "MetF", "hmdb_id": "HMDB00006", "reaction": "R_F_prod_1", "gene": "G_prodF", "direction": "product"},
    {"metabolite": "MetF", "hmdb_id": "HMDB00006", "reaction": "R_F_sub_1", "gene": "G_consA", "direction": "substrate"},
    {"metabolite": "MetD", "hmdb_id": "HMDB00004", "reaction": "R_D_sub_1", "gene": "G_consA", "direction": "substrate"},
    {"metabolite": "MetC", "hmdb_id": "HMDB00003", "reaction": "R_C_prod_1", "gene": "G_missing", "direction": "product"},
])

print("Reaction Table:")
display(reaction_table)

Reaction Table:


,metabolite,hmdb_id,reaction,gene,direction
0,MetA,HMDB00001,R_A_prod_1,G_prodA1,product
1,MetA,HMDB00001,R_A_prod_1,G_prodA2,product
2,MetA,HMDB00001,R_A_prod_2,G_prodA3,product
3,MetA,HMDB00001,R_A_sub_1,G_consA,substrate
4,MetA,HMDB00001,R_A_trans_1,G_transA1,exporter
5,MetA,HMDB00001,R_A_trans_1,G_transA2,exporter
6,MetB,HMDB00002,R_B_prod_1,"G_prodB, G_prodA1",product
7,MetF,HMDB00006,R_F_prod_1,G_prodF,product
8,MetF,HMDB00006,R_F_sub_1,G_consA,substrate
9,MetD,HMDB00004,R_D_sub_1,G_consA,substrate


## 5. 调用 cellmesh.compute_metabolite_availability

In [69]:
# 5.1 调用 API
result = cellmesh.compute_metabolite_availability(
    adata,
    reaction_table,
    celltype_col="cell_type",
    return_intermediates=True
)

In [ ]:
result.keys()

In [71]:
result['metadata']

,,has_product,has_substrate,has_exporter,n_product_reactions,n_substrate_reactions,n_exporter_reactions
metabolite,hmdb_id,,,,,,
MetA,HMDB00001,True,True,True,2,1,1
MetB,HMDB00002,True,False,False,1,0,0
MetF,HMDB00006,True,True,False,1,1,0


In [72]:
result['pseudobulk']

,G_prodA1,G_prodA2,G_prodA3,G_consA,G_transA1,G_transA2,G_prodB,G_prodF,G_noise
T_cell,1.333333,1.00,0.700000,0.233333,2.533333,1.5,0.100000,1.2,0.50
Macrophage,1.033333,2.80,0.533333,4.000000,0.800000,0.4,0.966667,0.5,0.70
Fibroblast,0.450000,0.65,0.250000,0.850000,0.350000,0.2,4.100000,2.6,0.95


In [73]:
display(result['P'])

,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetA,HMDB00001,1.860247,2.313022,0.796771
MetB,HMDB00002,0.602082,0.999722,1.719375
MetF,HMDB00006,1.200000,0.500000,2.600000


In [74]:
print("\nP (Product scores):")
display(result['P'].round(3))


P (Product scores):


,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetA,HMDB00001,1.860,2.313,0.797
MetB,HMDB00002,0.602,1.000,1.719
MetF,HMDB00006,1.200,0.500,2.600


## 6. 展示中间结果 P, C, E

In [75]:
print("\nP (Product scores):")
display(result['P'].round(3))

print("\nC (Substrate scores):")
display(result['C'].round(3))

print("\nE (Transporter scores):")
display(result['E'].round(3))


P (Product scores):


,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetA,HMDB00001,1.860,2.313,0.797
MetB,HMDB00002,0.602,1.000,1.719
MetF,HMDB00006,1.200,0.500,2.600



C (Substrate scores):


,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetA,HMDB00001,0.233,4.0,0.85
MetB,HMDB00002,0.000,0.0,0.00
MetF,HMDB00006,0.233,4.0,0.85



E (Transporter scores):


,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetA,HMDB00001,1.972,0.587,0.273
MetB,HMDB00002,0.000,0.000,0.000
MetF,HMDB00006,0.000,0.000,0.000


In [ ]:
print("\nP contrast and positive production anchor:")
display(result["P_contrast"].round(3))
display(result["P_plus"].round(3))

print("\nRelative consumption support:")
display(result["relative_consumption_support"].round(3))

print("\nExporter positive contrast:")
display(result["E_plus"].round(3))

print("\nSender scores:")
display(result["availability"].round(3))

print("\nMetadata:")
display(result["metadata"])


## 7. 展示 Metabolite Availability

In [77]:
print("\nMetabolite Availability (metabolite × cell_type):")
availability = result['availability']
display(availability.round(4))


Metabolite Availability (metabolite × cell_type):


,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetA,HMDB00001,0.7014,0.0000,0.000
MetB,HMDB00002,0.0000,0.2865,0.805
MetF,HMDB00006,0.3000,0.0000,0.823


## 8. Dense vs Sparse 一致性测试

In [78]:
# 8.1 创建 sparse AnnData
adata_sparse = SimpleAnnData(
    X=sparse.csr_matrix(X),
    var_names=genes,
    obs={"cell_type": cell_types}
)

# 8.2 用 sparse 输入计算
result_sparse = cellmesh.compute_metabolite_availability(
    adata_sparse,
    reaction_table,
    celltype_col="cell_type",
    return_intermediates=True
)

# 8.3 比较结果
print("\nDense vs Sparse availability difference:")
diff = availability - result_sparse['availability']
display(diff.round(10))

# 8.4 断言一致性
pd.testing.assert_frame_equal(
    availability,
    result_sparse['availability'],
    atol=1e-10
)
print("\n✓ Dense vs Sparse 输入结果一致！")


Dense vs Sparse availability difference:


,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetA,HMDB00001,0.0,0.0,0.0
MetB,HMDB00002,0.0,0.0,0.0
MetF,HMDB00006,0.0,0.0,0.0



✓ Dense vs Sparse 输入结果一致！


In [79]:
# 取前 5 个细胞和前 5 个基因
print(adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X)

[[4.  1.  0.7 0.2 2.5 1.5 0.1 1.2 0.5]
 [0.  1.2 0.8 0.3 2.7 1.4 0.  1.1 0.4]
 [0.  0.8 0.6 0.2 2.4 1.6 0.2 1.3 0.6]
 [1.  2.8 0.5 4.  0.8 0.4 1.  0.5 0.7]
 [1.2 3.  0.5 4.2 0.7 0.3 0.8 0.6 0.8]
 [0.9 2.6 0.6 3.8 0.9 0.5 1.1 0.4 0.6]
 [0.5 0.6 0.2 0.8 0.3 0.2 4.  2.5 0.9]
 [0.4 0.7 0.3 0.9 0.4 0.2 4.2 2.7 1. ]]


In [80]:
adata_sparse.obs

,cell_type
0,T_cell
1,T_cell
2,T_cell
3,Macrophage
4,Macrophage
5,Macrophage
6,Fibroblast
7,Fibroblast


In [81]:
availability

,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetA,HMDB00001,0.701385,0.000045,0.000000
MetB,HMDB00002,0.000000,0.286491,0.804984
MetF,HMDB00006,0.300000,0.000000,0.823037


In [82]:
result_sparse['availability']

,,T_cell,Macrophage,Fibroblast
metabolite,hmdb_id,,,
MetA,HMDB00001,0.701385,0.000045,0.000000
MetB,HMDB00002,0.000000,0.286491,0.804984
MetF,HMDB00006,0.300000,0.000000,0.823037


## 9. 用 toy expected result 做断言验证包内方法

In [ ]:
print("\n" + "=" * 70)
print("Running Toy Assertion Tests")
print("=" * 70)

pseudobulk = result["pseudobulk"]
print("\nPseudobulk:")
display(pseudobulk.round(3))

metA_idx = next(idx for idx in result["P"].index if idx[0] == "MetA")
print(f"\nTesting MetA at index: {metA_idx}")

from scipy.stats.mstats import gmean

genes_r1 = ["G_prodA1", "G_prodA2"]
expr_r1 = pseudobulk[genes_r1].values + 1
gmean_r1 = gmean(expr_r1, axis=1) - 1
expr_r2 = pseudobulk["G_prodA3"].values.reshape(-1, 1) + 1
gmean_r2 = gmean(expr_r2, axis=1) - 1

expected_metA_P = pd.Series(gmean_r1 + gmean_r2, index=pseudobulk.index)
actual_metA_P = result["P"].loc[metA_idx]
pd.testing.assert_series_equal(expected_metA_P, actual_metA_P, check_names=False, atol=1e-10)
print("✓ Multi-gene reaction uses geometric mean; multi-reaction sums")

metB_idx = next(idx for idx in result["P"].index if idx[0] == "MetB")
assert not result["metadata"].loc[metB_idx, "has_substrate"]
assert not result["metadata"].loc[metB_idx, "has_exporter"]
assert np.allclose(result["relative_consumption_support"].loc[metB_idx].values, 0.0)
assert np.allclose(result["E_plus"].loc[metB_idx].values, 0.0)
print("✓ Missing consumption and exporter priors are neutral")

included_mets = [idx[0] for idx in availability.index]
print(f"\nIncluded metabolites: {included_mets}")
assert "MetD" not in included_mets, "MetD should be skipped"
assert "MetC" not in included_mets, "MetC should be skipped"
print("✓ MetD (no product) and MetC (missing genes) are skipped")

print(f"\nAvailability shape: {availability.shape}")
assert availability.shape[1] == 3, "Should have 3 cell types"
print("✓ Availability shape is correct")

print("\n" + "=" * 70)
print("✓ All Toy Assertion Tests Passed!")
print("=" * 70)
